# 강의 03 · 실습 2 — 랭체인 기초 · (2-2) 빈칸 채우기 II

## 1. 문제상황

- 구름월드 놀이공원 안내 프로그램을 맡은 개발자는 모델을 부르는 코드를 손으로 한 줄씩 짭니다.
- FAQ를 지시문에 붙이고, 「반드시 JSON 하나만 출력하라」는 형식 문장을 넣고, 돌아온 답에서 코드 펜스를 벗기고, JSON으로 파싱하고, 필드가 다 있는지 검사하고, 실패하면 3회까지 다시 부릅니다.
- 질문 한 종류에 코드가 40줄을 넘고, 새 안내 기능을 하나 더할 때마다 같은 40줄을 다시 씁니다.
- 형식 문장을 조금만 바꿔도 파싱이 깨져, 어디가 잘못됐는지 찾는 데 시간이 듭니다.

## 2. 문제와 목표

- **문제**: 지시문 조립, 출력 형식 강제, 파싱, 필드 검증, 재시도를 전부 손으로 짜므로 코드가 길고 깨지기 쉽습니다.
- **목표**
  - 자주 쓰는 부품 세 개를 선언하고 파이프 기호로 이어 붙여, 같은 안내 프로그램을 짧게 다시 만듭니다.
    - 세 부품: 출력 스키마(`FaqAnswer` — `topic`·`answer` 두 칸), 모델(`llm`), 프롬프트 템플릿(`prompt`)
  - 결과는 파싱 없이 파이썬 객체로 받고, 필요하면 조각으로도 받습니다.
- **목표 달성 여부의 판정 기준**:
  - 운영시간 질문과 환불 질문을 넣었을 때 두 답이 모두 `FaqAnswer` 객체로 돌아오고 `topic`·`answer` 칸이 채워져 있으며,
  - 조각 실행에서 조각이 2개 이상 차례로 찍히는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex02_s1_diagram.svg)

## 4. 단계별 요구사항

1. **출력 스키마를 선언합니다.**
    - `topic`(주제)과 `answer`(답변) 두 문자열 칸을 가지는 `FaqAnswer` 클래스를 `BaseModel`을 상속해 선언합니다.
2. **모델을 초기화합니다.**
    - `init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")`로 모델 부품 `llm`을 만듭니다.
3. **프롬프트 템플릿을 선언합니다.**
    - `ChatPromptTemplate.from_messages`로 시스템 메시지(「너는 시설 안내 담당자다. 아래 FAQ만 근거로 답한다.」와 `{faq}` 빈칸)와 사용자 메시지(`{question}` 빈칸)를 가지는 템플릿 `prompt`를 만듭니다.
4. **부품을 조립합니다.**
    - `prompt | llm.with_structured_output(FaqAnswer)`로 체인 `chain`을 만듭니다.
    - `with_structured_output`이 형식 강제·파싱·검증을 대신합니다.
5. **실행합니다.**
    - `chain.invoke({"faq": FAQ_CONTEXT, "question": 질문})`으로 운영시간 질문과 환불 질문을 차례로 넣어 `topic`과 `answer`를 출력합니다.
    - 그 다음 `streaming=True`로 만든 모델과 같은 템플릿을 `prompt | llm_stream`으로 이어 텍스트 체인을 만들고, `chain.stream`으로 「환불 규정을 자세히 설명해 주세요.」의 답을 조각으로 받아 화면에 이어서 찍고 조각 수를 출력합니다.

## 5. 코드 골격 — LangChain 체인 3단

랭체인으로 체인을 세우는 순서는 다음 세 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 세 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 부품 선언 | 모델·프롬프트·출력 파서를 각각 하나씩 선언합니다 | `class FaqAnswer(BaseModel)`, `init_chat_model(...)`, `ChatPromptTemplate.from_messages` | 1, 2, 3 |
| ② 조립 | 부품을 파이프 기호로 한 줄에 잇습니다 | `prompt | llm.with_structured_output(FaqAnswer)` | 4 |
| ③ 실행 | 입력을 넣어 체인을 돌리고, 필요하면 조각으로 받습니다 | `chain.invoke(...)`, `chain.stream(...)` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
# 여기에 단계 0(라이브러리 불러오기, .env 읽기)을 작성합니다.

### 단계 ① — 부품 선언 (요구사항 1, 2, 3)

- 출력 스키마는 모델이 돌려줄 답의 모양입니다. 손 코드에서 「형식 문장 + JSON 파싱 + 필드 검증」이 하던 일을 클래스 선언 하나가 맡습니다.
- 모델 부품은 `init_chat_model`로 만듭니다. 프롬프트 템플릿은 `{faq}`·`{question}` 빈칸를 가진 메시지 틀입니다. 빈칸 이름은 실행할 때 넣는 딕셔너리의 키와 같아야 합니다.

In [ ]:
# 여기에 단계 ①(FAQ 문자열, 출력 스키마, 모델, 프롬프트 템플릿 선언)을 작성합니다.

### 단계 ② — 조립 (요구사항 4)

- 파이프 기호 `|`가 앞 부품의 출력을 뒤 부품의 입력으로 넘깁니다.
- `with_structured_output(FaqAnswer)`는 모델에 「이 모양으로 답하라」를 붙이고, 돌아온 답을 `FaqAnswer` 객체로 바꿔 줍니다. 손 코드의 형식 문장·파싱·검증·재시도가 이 한 줄로 접힙니다.

In [ ]:
# 여기에 단계 ②(체인 조립)를 작성합니다.

### 단계 ③ — 실행 (요구사항 5)

- `invoke`는 완성된 결과 하나를 돌려줍니다. 입력은 템플릿 빈칸 이름을 키로 하는 딕셔너리입니다.
- `stream`은 조각의 연속을 돌려줍니다. 조각을 받으려면 모델을 `streaming=True`로 만들고, 구조화 출력 없이 텍스트 체인으로 잇습니다. 체인의 부품과 순서는 그대로이고 부르는 방법만 다릅니다.

In [ ]:
# 여기에 단계 ③(invoke 두 질문, stream 텍스트 체인)을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 출력에서 스키마 칸이 `['topic', 'answer']`이고 템플릿 빈칸이 `['faq', 'question']`입니다.
2. 두 질문의 답이 `topic=…| answer=…` 형식으로 찍힙니다. 파싱 코드 없이 `out.topic`·`out.answer`로 값을 꺼냈습니다. 답의 내용은 FAQ 문장에 근거합니다.
3. `[stream]` 아래에 답이 이어서 찍히고 마지막에 `조각 수 = N`이 2 이상입니다. 같은 템플릿과 모델로 부르는 방법만 바꿨습니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 `lec03_ex02_s1.ipynb`와 대조해 채운 빈칸을 고칩니다.